In [2]:
import pandas as pd
import numpy as np

np.set_printoptions(threshold=np.inf)

In [3]:
# Loading guest data from the restaurant
guest_df = pd.read_csv("../data/WEEVA_GUESTS.csv")
guest_df.head(5)
# guest_df.shape

,DATE,GUESTS
0,11/1/2018,0
1,11/2/2018,0
2,11/3/2018,0
3,11/4/2018,0
4,11/5/2018,108


In [4]:
# Loading weather data by time frame
weather_nov_2018_may_2021_df       = pd.read_csv("../data/Groningen 2018-11-01 to 2021-05-31.csv")
weather_june_2021_december_2023_df = pd.read_csv("../data/Groningen 2021-06-01 to 2023-12-31.csv")
weather_jan_2024_april_2025_df     = pd.read_csv("../data/Groningen 2024-01-01 to 2025-04-28.csv")

# Combining the weather datasets into one dataframe
weather_df = pd.concat([weather_nov_2018_may_2021_df, 
                        weather_june_2021_december_2023_df, 
                        weather_jan_2024_april_2025_df], 
                        ignore_index=True)

weather_df.head()
weather_df.shape

(2371, 33)

In [5]:
# Loading data on holidays and calendar dates
school_holidays_df           = pd.read_csv("../data/groningen_school_holidays_boolean.csv")
public_holidays_groningen_df = pd.read_csv("../data/public_holidays_2018_2025.csv")
public_holidays_germany_df   = pd.read_csv("../data/public_holidays_germany_2018_2025.csv")
calendar_df                  = pd.read_csv("../data/dates_with_weekdays.csv")

calendar_df.head()

,Date,DayOfWeek,IsWeekend
0,2018-11-01,Thursday,False
1,2018-11-02,Friday,False
2,2018-11-03,Saturday,True
3,2018-11-04,Sunday,True
4,2018-11-05,Monday,False


In [6]:
school_holidays_df.tail()
school_holidays_df.shape
# school_holidays_df.dtypes

(3035, 2)

In [7]:
school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])

start_date = '2018-11-01'
end_date = '2025-04-28'

# Filter to keep only dates within the desired range
school_holidays_df = school_holidays_df[
    (school_holidays_df['Date'] >= start_date) &
    (school_holidays_df['Date'] <= end_date)
]

# Remove duplicate dates, keeping the last occurrence
school_holidays_df = school_holidays_df.drop_duplicates(subset='Date', keep='last')

school_holidays_bool_df = pd.DataFrame(school_holidays_df)
# Convert 'Yes'/'No' to True/False in a specific column (e.g., 'IsHoliday')
school_holidays_bool_df['IsHoliday'] = school_holidays_bool_df['IsHoliday'].map({"Yes": True, "No": False})

school_holidays_bool_df.shape

C:\Users\Matei\AppData\Local\Temp\ipykernel_436\819521641.py:1: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])


(2371, 2)

In [8]:
# 9 entries out of our date range for groningen
# public_holidays_groningen_df.head(50)
# public_holidays_groningen_df.tail(20)

public_holidays_groningen_df["Holiday"].unique()
# Only 9 unique holidays, but inconsistant naming () 
# e.g: 'Koningsdag (National Holiday)' / 'Koningsdag (National Day)'
# public_holidays_groningen_df["Holiday"].unique().shape

public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                       ['Holiday'].str.strip()

# public_holidays_groningen_df["Holiday"].unique()

name_map = {
    "New Year": "New Year's Day",
    "Koningsdag (National Holiday)": "King\'s Day",
     "Koningsdag (National Day)": "King\'s Day",
     "St. Stephen's Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_groningen_df["Holiday"].unique()
# # Only 7 unique holidays after filtering
# public_holidays_groningen_df["Holiday"].unique().shape

# public_holidays_groningen_df.head()


array(["New Year's Day", 'Easter Monday', "King's Day", 'Ascension Day',
       'Whit Monday', 'Christmas', 'Second Christmas Day'], dtype=object)

In [9]:
public_holidays_germany_df["Holiday"].unique()

public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                       ['Holiday'].str.strip()

name_map = {
    "New Years Day": "New Year's Day",
    "Christmas Day": "Christmas",
    "Boxing Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_germany_df["Holiday"].unique()


array(["New Year's Day", 'Good Friday', 'Easter Monday', 'May Day',
       'Ascension Day', 'Whit Monday', 'Day of German Unity', 'Christmas',
       'Second Christmas Day'], dtype=object)

In [10]:
# Setting the date column to the right data type
public_holidays_groningen_df["Date"] = pd.to_datetime(
                                        public_holidays_groningen_df["Date"],
                                        format="%d.%m.%Y")
public_holidays_germany_df["Date"] = pd.to_datetime(
                                        public_holidays_germany_df["Date"],
                                        format="%d.%m.%Y")

# Combine all the holidays
combined_holidays_df = pd.concat([public_holidays_groningen_df,
                                   public_holidays_germany_df],
                                    ignore_index=True)

combined_holidays_df["Holiday"].unique().shape

combined_holidays_df['is_holiday'] = 1


# Drop duplicates
combined_holidays_df = combined_holidays_df.pivot_table(
    index='Date',
    columns='Holiday',
    values='is_holiday',
    fill_value=0
).reset_index()

combined_holidays_df.shape

# Defining a range of dates for the full holiday dataframe
date_range = pd.date_range(start='2018-11-01', end='2025-04-28', freq='D')

# Create a new DataFrame with that full date range
full_date_range_df = pd.DataFrame({'Date': date_range})

# Merge on Date — left join to preserve full date range
merged_df = full_date_range_df.merge(combined_holidays_df,
                                     on='Date',
                                     how='left')

# # Fill NaN's 
final_holiday_df = merged_df.fillna('0').astype({col: 'int' for col in \
                                                     merged_df.columns \
                                                        if col != 'Date'})

final_holiday_df.head()
final_holiday_df.columns




Index(['Date', 'Ascension Day', 'Christmas', 'Day of German Unity',
       'Easter Monday', 'Good Friday', 'King's Day', 'May Day',
       'New Year's Day', 'Second Christmas Day', 'Whit Monday'],
      dtype='object')

In [11]:
calendar_df.head()
calendar_df.shape

(2371, 3)

In [38]:
# Loading data on number of items ordered in the restaurant
course_data_2018_2022_df = pd.read_csv("../data/Weeva_data_2018-2022.csv")
course_data_2023_2025_df = pd.read_csv("../data/Weeva_Data_2023-x.csv")

course_data_df = pd.concat([course_data_2018_2022_df, 
                            course_data_2023_2025_df], 
                            ignore_index=True)

# print(str(course_data_df["Article"].unique()).lower())
# print(course_data_df["Article"].head())
print(course_data_df.head())

print(course_data_df[course_data_df['Article'].str.lower() == 'groningse poffert']\
                                            ['Sold articles amount'].sum())
print(course_data_df[course_data_df['Article'].str.lower() == 'poffert menu']\
                                            ['Sold articles amount'].sum())
print(course_data_df[course_data_df['Article'].str.lower() == 'groninger poffert tb']\
                                            ['Sold articles amount'].sum())
print(course_data_df[course_data_df['Article'].str.lower() == 'wt- poffert']\
                                            ['Sold articles amount'].sum())
print(course_data_df[course_data_df['Article'].str.lower() == 'poffert afhaal']\
                                            ['Sold articles amount'].sum())

         Date             Article  Sold articles amount
0  27-11-2018       Heineken 25cl                    31
1  27-11-2018     Chardonnay Glas                     4
2  27-11-2018  3 Gang Arrangement                     1
3  27-11-2018       Heineken 22cl                     2
4  27-11-2018           Schnitzel                     8
10181
222
186
2
51
